# Deep Q-Network (DQN) with PyTorch

When the state space is large or continuous, we approximate $Q(s,a)$ with a neural network. This notebook:
1. Builds a **DQN agent** with experience replay and target network
2. Trains on OpenAI Gymnasium's **CartPole-v1**
3. Analyses training stability

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print('PyTorch not installed -- pip install torch')

try:
    import gymnasium as gym
    HAS_GYM = True
except ImportError:
    HAS_GYM = False
    print('gymnasium not installed -- pip install gymnasium')

%matplotlib inline

## 1. DQN Architecture

Key innovations from Mnih et al. (2015):
- **Experience replay**: store transitions $(s,a,r,s')$ in a buffer, sample mini-batches
- **Target network**: a slowly-updated copy of $Q$ to stabilise the TD target

Loss: $L = \mathbb{E}\left[ \left( r + \gamma \max_{a'} Q_{\text{target}}(s',a') - Q(s,a) \right)^2 \right]$

In [ ]:
if HAS_TORCH:
    class QNetwork(nn.Module):
        def __init__(self, state_dim, action_dim, hidden=128):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(state_dim, hidden),
                nn.ReLU(),
                nn.Linear(hidden, hidden),
                nn.ReLU(),
                nn.Linear(hidden, action_dim)
            )
        
        def forward(self, x):
            return self.net(x)
    
    class ReplayBuffer:
        def __init__(self, capacity=10000):
            self.buffer = deque(maxlen=capacity)
        
        def push(self, state, action, reward, next_state, done):
            self.buffer.append((state, action, reward, next_state, done))
        
        def sample(self, batch_size):
            batch = random.sample(self.buffer, batch_size)
            states, actions, rewards, next_states, dones = zip(*batch)
            return (np.array(states), np.array(actions), np.array(rewards, dtype=np.float32),
                    np.array(next_states), np.array(dones, dtype=np.float32))
        
        def __len__(self):
            return len(self.buffer)
    
    print('QNetwork and ReplayBuffer defined.')

In [ ]:
if HAS_TORCH and HAS_GYM:
    class DQNAgent:
        def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99,
                     epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=500):
            self.action_dim = action_dim
            self.gamma = gamma
            self.epsilon_start = epsilon_start
            self.epsilon_end = epsilon_end
            self.epsilon_decay = epsilon_decay
            self.steps = 0
            
            self.q_net = QNetwork(state_dim, action_dim)
            self.target_net = QNetwork(state_dim, action_dim)
            self.target_net.load_state_dict(self.q_net.state_dict())
            self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
            self.buffer = ReplayBuffer()
        
        def epsilon(self):
            return self.epsilon_end + (self.epsilon_start - self.epsilon_end) * \
                   np.exp(-self.steps / self.epsilon_decay)
        
        def select_action(self, state):
            self.steps += 1
            if np.random.rand() < self.epsilon():
                return np.random.randint(self.action_dim)
            with torch.no_grad():
                q = self.q_net(torch.FloatTensor(state).unsqueeze(0))
                return q.argmax(dim=1).item()
        
        def train_step(self, batch_size=64):
            if len(self.buffer) < batch_size:
                return None
            states, actions, rewards, next_states, dones = self.buffer.sample(batch_size)
            
            states_t = torch.FloatTensor(states)
            actions_t = torch.LongTensor(actions).unsqueeze(1)
            rewards_t = torch.FloatTensor(rewards).unsqueeze(1)
            next_states_t = torch.FloatTensor(next_states)
            dones_t = torch.FloatTensor(dones).unsqueeze(1)
            
            # Current Q values
            q_values = self.q_net(states_t).gather(1, actions_t)
            
            # Target Q values
            with torch.no_grad():
                max_next_q = self.target_net(next_states_t).max(1, keepdim=True)[0]
                target = rewards_t + self.gamma * max_next_q * (1 - dones_t)
            
            loss = nn.MSELoss()(q_values, target)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            return loss.item()
        
        def update_target(self):
            self.target_net.load_state_dict(self.q_net.state_dict())
    
    print('DQNAgent defined.')

In [ ]:
if HAS_TORCH and HAS_GYM:
    env = gym.make('CartPole-v1')
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    agent = DQNAgent(state_dim, action_dim)
    
    n_episodes = 300
    target_update_freq = 10
    rewards_history = []
    
    for ep in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0
        
        for t in range(500):
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            agent.buffer.push(state, action, reward, next_state, float(done))
            agent.train_step()
            state = next_state
            total_reward += reward
            if done:
                break
        
        if (ep + 1) % target_update_freq == 0:
            agent.update_target()
        
        rewards_history.append(total_reward)
        if (ep + 1) % 50 == 0:
            avg = np.mean(rewards_history[-50:])
            print(f"Episode {ep+1:3d}  avg_reward={avg:.1f}  epsilon={agent.epsilon():.3f}")
    
    env.close()

In [ ]:
if HAS_TORCH and HAS_GYM:
    smoothed = np.convolve(rewards_history, np.ones(20)/20, mode='valid')
    plt.plot(smoothed)
    plt.axhline(475, color='r', linestyle='--', label='Solved threshold')
    plt.xlabel('Episode')
    plt.ylabel('Total Reward (smoothed)')
    plt.title('DQN on CartPole-v1')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## Key Takeaways

- **DQN** uses a neural network to approximate $Q(s,a)$ for large state spaces.
- **Experience replay** breaks correlation between consecutive samples.
- **Target network** stabilises training by providing a fixed TD target.
- Extensions: Double DQN, Dueling DQN, Prioritised Experience Replay.

**Next:** Policy gradient methods (REINFORCE).